<h2 align='center' style='color:purple'>Finding best model and hyper parameter tunning using GridSearchCV</h2>

**For iris flower dataset in sklearn library, we are going to find out best model and best hyper parameters using GridSearchCV**

<img src='iris_petal_sepal.png' height=300 width=300 />

**Load iris flower dataset**

In [1]:
# imports 
import numpy as np
import pandas as pd
from sklearn import svm, datasets
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV
)

# algorithms
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [2]:
iris = datasets.load_iris()

In [3]:
import pandas as pd
df = pd.DataFrame(iris.data,columns=iris.feature_names)
df['flower'] = iris.target
df['flower'] = df['flower'].apply(lambda x: iris.target_names[x])
df[47:50]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),flower
47,4.6,3.2,1.4,0.2,setosa
48,5.3,3.7,1.5,0.2,setosa
49,5.0,3.3,1.4,0.2,setosa


<h3 style='color:blue'>Approach 1: Use train_test_split and manually tune parameters by trial and error</h3>

In [4]:
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.3)

In [5]:
model = svm.SVC(kernel='rbf',C=30,gamma='auto')
model.fit(X_train,y_train)
model.score(X_test, y_test)

0.9333333333333333

<h3 style='color:blue'>Approach 2: Use K Fold Cross validation</h3>

**Manually try suppling models with different parameters to cross_val_score function with 5 fold cross validation**

In [6]:
from sklearn.model_selection import cross_val_score
cross_val_score(svm.SVC(kernel='linear',C=10,gamma='auto'),iris.data, iris.target, cv=5)

array([1.        , 1.        , 0.9       , 0.96666667, 1.        ])

In [7]:
cross_val_score(svm.SVC(kernel='rbf',C=10,gamma='auto'),iris.data, iris.target, cv=5)

array([0.96666667, 1.        , 0.96666667, 0.96666667, 1.        ])

In [8]:
cross_val_score(svm.SVC(kernel='rbf',C=20,gamma='auto'),iris.data, iris.target, cv=5)

array([0.96666667, 1.        , 0.9       , 0.96666667, 1.        ])

**Above approach is tiresome and very manual. We can use for loop as an alternative**

In [9]:
kernels = ['rbf', 'linear']
C = [1,10,20]
avg_scores = {}
for kval in kernels:
    for cval in C:
        cv_scores = cross_val_score(svm.SVC(kernel=kval,C=cval,gamma='auto'),iris.data, iris.target, cv=5)
        avg_scores[kval + '_' + str(cval)] = np.average(cv_scores)

avg_scores

{'rbf_1': 0.9800000000000001,
 'rbf_10': 0.9800000000000001,
 'rbf_20': 0.9666666666666668,
 'linear_1': 0.9800000000000001,
 'linear_10': 0.9733333333333334,
 'linear_20': 0.9666666666666666}

**From above results we can say that rbf with C=1 or 10 or linear with C=1 will give best performance**

<h3 style='color:blue'>Approach 3: Use GridSearchCV</h3>

**GridSearchCV does exactly same thing as for loop above but in a single line of code**

In [10]:
from sklearn.model_selection import GridSearchCV
svc = svm.SVC(gamma='auto')

In [11]:

clf = GridSearchCV(svc, {
    'C': [1,10,20],
    'kernel': ['rbf','linear']
}, cv=5, return_train_score=True)
clf.fit(iris.data, iris.target)
clf.cv_results_

{'mean_fit_time': array([0.00260053, 0.00279665, 0.00248566, 0.00200086, 0.00216641,
        0.00099988]),
 'std_fit_time': array([8.03515190e-04, 3.98803863e-04, 4.61548482e-04, 3.24599694e-06,
        7.53887684e-04, 3.16082257e-06]),
 'mean_score_time': array([0.00140138, 0.00119662, 0.00143189, 0.00137687, 0.0009798 ,
        0.00059881]),
 'std_score_time': array([8.12457021e-04, 3.99764758e-04, 4.69450684e-04, 5.11151485e-04,
        4.31479930e-05, 4.88935034e-04]),
 'param_C': masked_array(data=[1, 1, 10, 10, 20, 20],
              mask=[False, False, False, False, False, False],
        fill_value=999999),
 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear'],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'C': 1, 'kernel': 'rbf'},
  {'C': 1, 'kernel': 'linear'},
  {'C': 10, 'kernel': 'rbf'},
  {'C': 10, 'kernel': 'linear'},
  {'C': 20, 'kernel': 'rbf'},
  {'C': 20

In [12]:
df = pd.DataFrame(clf.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,...,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,0.002601,0.000804,0.001401,0.000812,1,rbf,"{'C': 1, 'kernel': 'rbf'}",0.966667,1.0,0.966667,...,0.980000,0.016330,1,0.983333,0.983333,0.991667,0.983333,0.975000,0.983333,0.005270
1,0.002797,0.000399,0.001197,0.000400,1,linear,"{'C': 1, 'kernel': 'linear'}",0.966667,1.0,0.966667,...,0.980000,0.016330,1,0.975000,0.975000,0.991667,0.983333,0.983333,0.981667,0.006236
2,0.002486,0.000462,0.001432,0.000469,10,rbf,"{'C': 10, 'kernel': 'rbf'}",0.966667,1.0,0.966667,...,0.980000,0.016330,1,0.975000,0.966667,0.991667,0.983333,0.966667,0.976667,0.009718
3,0.002001,0.000003,0.001377,0.000511,10,linear,"{'C': 10, 'kernel': 'linear'}",1.000000,1.0,0.900000,...,0.973333,0.038873,4,0.966667,0.966667,0.991667,0.991667,0.975000,0.978333,0.011304
4,0.002166,0.000754,0.000980,0.000043,20,rbf,"{'C': 20, 'kernel': 'rbf'}",0.966667,1.0,0.900000,...,0.966667,0.036515,5,0.966667,0.975000,0.991667,0.991667,0.975000,0.980000,0.010000
5,0.001000,0.000003,0.000599,0.000489,20,linear,"{'C': 20, 'kernel': 'linear'}",1.000000,1.0,0.900000,...,0.966667,0.042164,6,0.975000,0.966667,0.991667,1.000000,0.975000,0.981667,0.012247


In [13]:
df[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.980000
1,1,linear,0.980000
2,10,rbf,0.980000
3,10,linear,0.973333
4,20,rbf,0.966667
5,20,linear,0.966667


In [14]:
def set_model_class_properties(model_obj, property_data:dict):
    for property_name, property_value in property_data.items():
        setattr(model_obj, property_name, property_value)
            
    return model_obj

In [15]:
svc_new = set_model_class_properties(svc,clf.best_params_)

In [16]:
svc_new.fit(X_train,y_train)

SVC(C=1, gamma='auto')

In [17]:
svc_new

SVC(C=1, gamma='auto')

In [18]:
clf.best_params_

{'C': 1, 'kernel': 'rbf'}

In [19]:
clf.best_score_

0.9800000000000001

In [20]:
dir(clf)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__sklearn_clone__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_build_request_for_signature',
 '_check_feature_names',
 '_check_n_features',
 '_check_refit_for_multimetric',
 '_doc_link_module',
 '_doc_link_template',
 '_doc_link_url_param_generator',
 '_estimator_type',
 '_format_results',
 '_get_default_requests',
 '_get_doc_link',
 '_get_metadata_request',
 '_get_param_names',
 '_get_routed_params_for_fit',
 '_get_scorers',
 '_get_tags',
 '_more_tags',
 '_parameter_constraints',
 '_repr_html_',
 '_repr_html_inner',
 '_repr_mimebundle_',
 '_required_parameters',
 '_run

**Use RandomizedSearchCV to reduce number of iterations and with random combination of parameters. This is useful when you have too many parameters to try and your training time is longer. It helps reduce the cost of computation**

In [21]:
clf.cv_results_.keys()

dict_keys(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time', 'param_C', 'param_kernel', 'params', 'split0_test_score', 'split1_test_score', 'split2_test_score', 'split3_test_score', 'split4_test_score', 'mean_test_score', 'std_test_score', 'rank_test_score', 'split0_train_score', 'split1_train_score', 'split2_train_score', 'split3_train_score', 'split4_train_score', 'mean_train_score', 'std_train_score'])

In [22]:
from dataclasses import dataclass, field
from collections import defaultdict
from typing import List

In [23]:
grid_search_results = clf.cv_results_
test_r2_mean = list(grid_search_results['mean_test_score'])
train_r2_mean = list(grid_search_results['mean_train_score'])
test_r2_std = list(grid_search_results['std_score_time'])
params = grid_search_results['params']

model_name = clf.estimator.__class__.__name__

In [24]:
params

[{'C': 1, 'kernel': 'rbf'},
 {'C': 1, 'kernel': 'linear'},
 {'C': 10, 'kernel': 'rbf'},
 {'C': 10, 'kernel': 'linear'},
 {'C': 20, 'kernel': 'rbf'},
 {'C': 20, 'kernel': 'linear'}]

In [25]:
@dataclass
class grid_model:
    """
    model_detail = {'model_name' : str, 'model_no' : str}
    metrics = {'val_r2_score' : val, 'val_r2_std' : val, 'overfit_gap' : val}
    parameters = {}
    """
    model_detail : dict = field(default_factory= lambda: defaultdict(str))
    metrics : dict = field(default_factory= lambda: defaultdict(float))
    parameters : dict = field(default_factory=dict)

In [26]:
num_grid_search = len(test_r2_mean)
Grid_Model_List : List[grid_model] = []
for model_no in range(num_grid_search):
    model_instance = grid_model()

    # model_detail
    model_instance.model_detail['model_name'] = model_name
    model_instance.model_detail['model_no'] = 'model_' + str(model_no)

    # metrics
    model_instance.metrics['val_r2_score'] = test_r2_mean[model_no]
    model_instance.metrics['val_r2_std'] = test_r2_std[model_no]
    model_instance.metrics['overfit_gap'] = train_r2_mean[model_no] - test_r2_mean[model_no]

    # parameters
    model_instance.parameters = params[model_no]

    # append to Grid_Model_List
    Grid_Model_List.append(model_instance)

In [27]:
def get_best_model_optimized(grid_models_list:List[grid_model], base_r2=0.6, overfit_gap=0.1)->grid_model:
    local_base_r2 = base_r2
    candidates = []
    count = 0
        
    # Filter valid models (R² ≥ base_r2)
    for grid_model in grid_models_list:
        r2_score = grid_model.metrics['val_r2_score']
        overfit = grid_model.metrics['overfit_gap']
            
        if r2_score >= local_base_r2 and overfit <= overfit_gap:
            local_base_r2 = r2_score
            candidates.append((grid_model, r2_score, overfit))
            count+=1
        
    if not candidates:
        raise ValueError(f"No valid model for {grid_model.model_detail['model_no']}: R²≥{base_r2}, gap≤{overfit_gap}")
        
    # PRIORITY 1: Highest R²
    # PRIORITY 2: Lowest overfit gap (tie-breaker)
    best_model_tuple = max(
        candidates, 
        key=lambda x: (x[1], -x[2])  # R² ASC, overfit DESC (negative = min)
    )
    best_grid_model = best_model_tuple[0]
    print(f"Run for loop: {count}")    
    return best_grid_model        

In [33]:
def get_best_model(grid_models_list:List[grid_model], BASE_R2=0.6, OVERFIT_GAP=0.1)->grid_model:
    """
    Check for best validation r2 score mean, if equal check for the least overfit gap
    """
    local_base_r2 = BASE_R2
    local_overfit_gap = OVERFIT_GAP
    best_model = None
    
    for model_instance in grid_models_list:
        model_r2 = model_instance.metrics['val_r2_score']
        model_overfit_gap = model_instance.metrics['overfit_gap']
        
        if (model_r2 > local_base_r2 and model_overfit_gap < OVERFIT_GAP) or (
            model_r2 == local_base_r2 and model_overfit_gap < local_overfit_gap):
            best_model = model_instance
            local_base_r2 = model_r2
            local_overfit_gap = model_overfit_gap

    return best_model
            

In [35]:
best_model = get_best_model(Grid_Model_List)
print(best_model.metrics)
print(best_model.parameters)

defaultdict(<class 'float'>, {'val_r2_score': 0.9800000000000001, 'val_r2_std': 0.000469450684211498, 'overfit_gap': -0.0033333333333332993})
{'C': 10, 'kernel': 'rbf'}


In [119]:
for model in Grid_Model_List:
    print(f"Overfit Gap : {model.metrics['overfit_gap']}")

Overfit Gap : 0.0033333333333330772
Overfit Gap : 0.0016666666666665941
Overfit Gap : -0.0033333333333332993
Overfit Gap : 0.004999999999999893
Overfit Gap : 0.013333333333333308
Overfit Gap : 0.015000000000000124


In [62]:
test_r2_mean

array([0.98      , 0.98      , 0.98      , 0.97333333, 0.96666667,
       0.96666667])

In [64]:
params

[{'C': 1, 'kernel': 'rbf'},
 {'C': 1, 'kernel': 'linear'},
 {'C': 10, 'kernel': 'rbf'},
 {'C': 10, 'kernel': 'linear'},
 {'C': 20, 'kernel': 'rbf'},
 {'C': 20, 'kernel': 'linear'}]

In [45]:
rs = RandomizedSearchCV(svm.SVC(gamma='auto'), {
        'C': [1,10,20],
        'kernel': ['rbf','linear']
    }, 
    cv=5, 
    return_train_score=False, 
    n_iter=2
)
rs.fit(iris.data, iris.target)
pd.DataFrame(rs.cv_results_)[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.98
1,1,linear,0.98


**How about different models with different hyperparameters?**

In [48]:


model_params = {
    'svm': {
        'model': svm.SVC(gamma='auto'),
        'params' : {
            'C': [1,10,20],
            'kernel': ['rbf','linear']
        }  
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params' : {
            'n_estimators': [1,5,10]
        }
    },
    'logistic_regression' : {
        'model': LogisticRegression(solver='liblinear',multi_class='auto'),
        'params': {
            'C': [1,5,10]
        }
    }
}


In [49]:
scores = []

for model_name, mp in model_params.items():
    clf =  GridSearchCV(mp['model'], mp['params'], cv=5, return_train_score=False)
    clf.fit(iris.data, iris.target)
    scores.append({
        'model': model_name,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })
    
df = pd.DataFrame(scores,columns=['model','best_score','best_params'])
df

C:\Users\Sreeman\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\Sreeman\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\Sreeman\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
C:\Users\Sreeman\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated i

,model,best_score,best_params
0,svm,0.980000,"{'C': 1, 'kernel': 'rbf'}"
1,random_forest,0.966667,{'n_estimators': 5}
2,logistic_regression,0.966667,{'C': 5}


In [122]:
from sklearn.tree import DecisionTreeRegressor

In [123]:
dtree = DecisionTreeRegressor(criterion="squared_error", max_depth=2)

In [126]:
dtree.__class__.__name__

'DecisionTreeRegressor'

**Based on above, I can conclude that SVM with C=1 and kernel='rbf' is the best model for solving my problem of iris flower classification**